<a href="https://colab.research.google.com/github/Shamita-Ramesh/GEN-AI/blob/main/GEN%20AI%20Exp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q "transformers<5" rouge_score evaluate

In [4]:
#Exp 1 (News Article Summarization) Dataset-"news_articles.csv"

import pandas as pd, evaluate, torch
from google.colab import files
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

files.upload()                                   # upload news_articles.csv
df = pd.read_csv("news_articles.csv")

name = "facebook/bart-large-cnn"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

# Input constraint: truncate articles to 1024 tokens
inputs = tok(df["article"].tolist(), max_length=1024, truncation=True, padding=True, return_tensors="pt")

# Output constraint: summary length between 20 and 60 tokens
with torch.no_grad():
    ids = model.generate(**inputs, max_length=60, min_length=20, num_beams=4, do_sample=False)
preds = tok.batch_decode(ids, skip_special_tokens=True)

for i in range(len(df)):
    print(f"\nArticle {i+1}")
    print("Generated Summary:", preds[i])
    print("Reference Summary:", df["highlights"][i])

rouge = evaluate.load("rouge")
scores = rouge.compute(predictions=preds, references=df["highlights"].tolist())
print("\nROUGE-1:", round(float(scores["rouge1"]), 4))
print("ROUGE-2:", round(float(scores["rouge2"]), 4))
print("ROUGE-L:", round(float(scores["rougeL"]), 4))

Saving news_articles.csv to news_articles (5).csv

Article 1
Generated Summary: The project is expected to cost about 2 billion dollars and will connect the northern suburbs with the main business district. Officials said the expansion would cut average commute times by nearly 25 percent and reduce road congestion. Construction is scheduled to begin early next year, subject to environmental clearances.
Reference Summary: City council approves a 40 km metro expansion costing 2 billion dollars to cut commute times and congestion.

Article 2
Generated Summary: Scientists at a leading research institute have developed a low-cost solar panel that can generate electricity even in cloudy conditions. In laboratory tests, the panel maintained over 80 percent of its efficiency on overcast days. The team hopes to begin commercial production within three years.
Reference Summary: Researchers build a low-cost perovskite solar panel that works efficiently on cloudy days, with production planned with

In [3]:
#Exp 2(Reviews) Dataset-"reviews.csv"

!rm -f reviews*.csv
import pandas as pd, torch
from google.colab import files
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report

up = files.upload()                               # upload the NEW reviews.csv
df = pd.read_csv(list(up.keys())[0])              # reads the file just uploaded
print(df.head())                                  # should show columns: review, label

name = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)

inputs = tok(df["review"].tolist(), truncation=True, max_length=128, padding=True, return_tensors="pt")
with torch.no_grad():
    probs = torch.softmax(model(**inputs).logits, dim=1)

df["predicted"] = [model.config.id2label[i.item()] for i in probs.argmax(1)]
df["confidence"] = probs.max(1).values.numpy().round(3)

df["short_review"] = df["review"].str[:55] + "..."
print(df[["short_review", "label", "predicted", "confidence"]].to_string())

print("\nAccuracy:", round(accuracy_score(df["label"], df["predicted"]), 3))
print(classification_report(df["label"], df["predicted"]))

print("\nMisclassified reviews:")
print(df[df["label"] != df["predicted"]][["review", "label", "predicted", "confidence"]].to_string())

Saving reviews.csv to reviews.csv
                                              review     label
0  The phone has an amazing camera and the batter...  POSITIVE
1  Absolutely loved this movie. The acting was br...  POSITIVE
2  The food was delicious and the staff were very...  POSITIVE
3  Great value for money. The laptop is fast and ...  POSITIVE
4  Excellent customer service. They solved my pro...  POSITIVE
                                                  short_review     label predicted  confidence
0   The phone has an amazing camera and the battery easily ...  POSITIVE  POSITIVE       0.999
1   Absolutely loved this movie. The acting was brilliant a...  POSITIVE  POSITIVE       1.000
2   The food was delicious and the staff were very friendly...  POSITIVE  POSITIVE       1.000
3   Great value for money. The laptop is fast and lightweig...  POSITIVE  POSITIVE       1.000
4   Excellent customer service. They solved my problem with...  POSITIVE  POSITIVE       1.000
5   The hotel room

In [4]:
#Exp 3(Question Answering system for biomedical queries) Dataset-"pubmed_abstracts.csv"

import pandas as pd
from google.colab import files
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

files.upload()                                    # upload pubmed_abstracts.csv
df = pd.read_csv("pubmed_abstracts.csv")
docs = (df["title"] + ". " + df["abstract"]).tolist()

# 1. Retriever: biomedical sentence-embedding model
retriever = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")
doc_emb = retriever.encode(docs, convert_to_tensor=True)

# 2. Generator: LLM (Flan-T5)
tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def rag_answer(question, k=2):
    hits = util.semantic_search(retriever.encode(question, convert_to_tensor=True), doc_emb, top_k=k)[0]
    context = " ".join(docs[h["corpus_id"]] for h in hits)
    prompt = f"Answer the clinical question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:"
    ids = tok(prompt, return_tensors="pt", max_length=512, truncation=True)
    out = llm.generate(**ids, max_new_tokens=80, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True), [df["doc_id"][h["corpus_id"]] for h in hits]

for q in ["What is the first-line drug for type 2 diabetes?",
          "Is aspirin recommended for primary prevention of heart disease?",
          "Which antibiotic is used first for outpatient pneumonia?",
          "Which anticoagulant is preferred for patients with mechanical heart valves?"]:
    ans, src = rag_answer(q)
    print(f"\nQ: {q}\nA: {ans}\nSources: {src}")

Saving pubmed_abstracts.csv to pubmed_abstracts (2).csv

Q: What is the first-line drug for type 2 diabetes?
A: Metformin
Sources: ['D01', 'D02']

Q: Is aspirin recommended for primary prevention of heart disease?
A: no
Sources: ['D03', 'D02']

Q: Which antibiotic is used first for outpatient pneumonia?
A: amoxicillin
Sources: ['D05', 'D02']

Q: Which anticoagulant is preferred for patients with mechanical heart valves?
A: Warfarin
Sources: ['D10', 'D03']


In [7]:
!pip install -q "transformers<5" faiss-cpu sentence-transformers

In [9]:
#Exp 4(Question Answer Summarization system for law cases) Dataset-"legal_cases.csv"

!rm -f legal_cases*.csv
import pandas as pd, faiss
from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

up = files.upload()                               # upload legal_cases.csv
df = pd.read_csv(list(up.keys())[0])
print(df.head(2))

# 1. FAISS index over case-law text
emb = SentenceTransformer("all-MiniLM-L6-v2")
vecs = emb.encode(df["text"].tolist(), normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(vecs.shape[1])          # inner product = cosine (normalized)
index.add(vecs)

# 2. Language model (T5)
tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def gen(prompt, max_t, min_t=5):
    ids = tok(prompt, return_tensors="pt", max_length=512, truncation=True)
    out = llm.generate(**ids, max_new_tokens=max_t, min_new_tokens=min_t, num_beams=4, no_repeat_ngram_size=3)
    return tok.decode(out[0], skip_special_tokens=True)

def legal_qa(q, k=2):
    _, I = index.search(emb.encode([q], normalize_embeddings=True).astype("float32"), k)
    cases = df.iloc[I[0]]
    ctx = " ".join(cases["text"])
    print("\nQuestion:", q)
    print("Retrieved cases:", cases["case_name"].tolist())
    print("Answer:", gen(f"Answer the legal question using the context.\nContext: {ctx}\nQuestion: {q}\nAnswer:", 60))
    print("Level 1 (one-line summary):", gen(f"Summarize in one sentence: {ctx}", 40, 10))
    print("Level 2 (detailed summary):", gen(f"Summarize in detail: {ctx}", 150, 60))

for q in ["Does a manufacturer owe a duty of care to the consumer?",
          "Can a minor enter into a valid contract in India?",
          "Is the right to privacy a fundamental right?",
          "What is the basic structure doctrine?"]:
    legal_qa(q)

Saving legal_cases.csv to legal_cases.csv
  case_id                                case_name  \
0     C01              Donoghue v Stevenson (1932)   
1     C02  Carlill v Carbolic Smoke Ball Co (1893)   

                                                text  
0  Mrs Donoghue drank ginger beer from an opaque ...  
1  A company advertised a reward of 100 pounds to...  

Question: Does a manufacturer owe a duty of care to the consumer?
Retrieved cases: ['Donoghue v Stevenson (1932)', 'Hadley v Baxendale (1854)']
Answer: the House of Lords held that a manufacturer owes a duty of care to the ultimate consumer of its products
Level 1 (one-line summary): A woman who fell ill from a bottle of ginger beer had a duty of care to her neighbour, the House of Lords.
Level 2 (detailed summary): A woman who fell ill after drinking a bottle of ginger beer had a duty of care to her neighbour, Lord Atkin, in a case based on the principle of the neighbour principle, the Court of Appeal has ruled in the ca

In [13]:
!pip install -q crewai

In [14]:
#Exp 5(Itinerary Planner) Dataset-"travel_options.csv"
#Create/Get Gemini API Keys from Gemini AI Studio

import os, pandas as pd
from getpass import getpass
from google.colab import files
from crewai import Agent, Task, Crew, Process, LLM

# LLM (Gemini). Model names get retired often, so copy the current Flash model name from AI Studio if this one errors
MODEL = "gemini/gemini-3.7-flash"
llm = LLM(model=MODEL, api_key=getpass("Enter Gemini API key: "), temperature=0.3)

up = files.upload()                               # upload travel_options.csv
df = pd.read_csv(list(up.keys())[0])

dest, days, budget = "Goa", 3, 15000              # change destination / days / budget (INR)
data = df[df["destination"] == dest].to_string(index=False)

def make_agent(role, goal):
    return Agent(role=role, goal=goal, backstory=f"You are an expert {role} for Indian travel.", llm=llm, verbose=True)

budgeter = make_agent("Budget Analyst", "Split the total budget across stay, food, transport and activities")
stay     = make_agent("Accommodation and Transport Planner", "Choose best-value hotel and local transport")
fun      = make_agent("Activities and Food Planner", "Choose top-rated activities and meals within budget")
lead     = make_agent("Itinerary Coordinator", "Combine all inputs into one cohesive itinerary within the total budget")

t1 = Task(description=f"Total budget is Rs {budget} for {days} days in {dest} for 1 traveller. Allocate the budget by category.",
          expected_output="Budget split by category", agent=budgeter)
t2 = Task(description=f"Using only this data, choose a hotel and transport within the allocated budget:\n{data}",
          expected_output="Chosen hotel and transport with costs", agent=stay, context=[t1])
t3 = Task(description=f"Using only this data, choose activities and meals within the allocated budget:\n{data}",
          expected_output="Chosen activities and meals with costs", agent=fun, context=[t1])
t4 = Task(description=f"Create a final {days}-day itinerary with cost per day and the total. The total must not exceed Rs {budget}.",
          expected_output="Day-wise itinerary with cost breakdown and total", agent=lead, context=[t1, t2, t3])

crew = Crew(agents=[budgeter, stay, fun, lead], tasks=[t1, t2, t3, t4], process=Process.sequential, verbose=True)
result = await crew.kickoff_async()
print("\n===== FINAL ITINERARY =====\n", result.raw)

Enter Gemini API key: ··········


Saving travel_options.csv to travel_options (3).csv


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9e76ac34-3ef4-457d-847c-31fd1edc1e8d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Total budget is Rs 15000 for 3 days in Goa for 1 traveller. Allocate the budget by category.             │
│  ID: c068c451-4fda-4de6-8d71-1b39111a980b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget Analyst                                                                                          │
│                                                                                                                 │
│  Task: Total budget is Rs 15000 for 3 days in Goa for 1 traveller. Allocate the budget by category.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Itinerary Coordinator                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 3-Day Comprehensive Goa Solo Itinerary                                                                       │
│                                                                                                                 │
│  **Traveller:** 1 Solo Traveller                                                                                │
│  **Duration:** 3 Days / 3 Nights                                                                                │
│  **Base Accommodation:** Backpacker Hostel Dorm near Calangute                                                  │
│  **Primary Transit:** Self-Drive Scooter Rental + Kadamba Airport Shuttle                                       │
│  **Total Budget Cap:** ₹15,000                                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## **Day 1: Coastal Forts, Dolphin Cruise & Beachside Dining**                                                 │
│                                                                                                                 │
│  ### **Schedule & Route**                                                                                       │
│  * **Morning (08:00 – 12:00): Arrival & Coastal Cruise**                                                        │
│    * Arrive in Goa and take the Kadamba EV AC airport shuttle to Calangute.                                     │
│    * Check into **Backpacker Hostel Dorm near Calangute** and pick up your rental scooter.                      │
│    * Head down to the shore for the **Dolphin Boat Trip** (1.5-hour marine sightseeing excursion along the      │
│  Sinquerim/Calangute coast).                                                                                    │
│  * **Afternoon (12:30 – 16:30): Authentic Goan Lunch & Relaxation**                                             │
│    * Relish an authentic **Local Fish Thali** at an acclaimed local coastal eatery.                             │
│    * Rest and unwind at the hostel or lounge along Calangute Beach.                                             │
│  * **Evening (16:30 – 21:00): Heritage Sunset & Seafood Dinner**                                                │
│    * Ride to **Fort Aguada**; explore the 17th-century Portuguese coastal bastion and enjoy sunset views over   │
│  the Arabian Sea.                                                                                               │
│    * Dine on fresh catch with your feet in the sand with a **Beach Shack Seafood Dinner**.                      │
│                                                                                                                 │
│  ### **Day 1 Cost Breakdown**                                                                                   │
│  | Item / Expense | Category | Cost (INR) |                                                                     │
│  | :--- | :--- | :--- |                                                                                         │
│  | Backpacker Hostel Dorm (Night 1) | Stay | ₹700 |    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create a final 3-day itinerary with cost per day and the total. The total must not exceed Rs 15000.      │
│  Agent: Itinerary Coordinator                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== FINAL ITINERARY =====
 # 3-Day Comprehensive Goa Solo Itinerary

**Traveller:** 1 Solo Traveller  
**Duration:** 3 Days / 3 Nights  
**Base Accommodation:** Backpacker Hostel Dorm near Calangute  
**Primary Transit:** Self-Drive Scooter Rental + Kadamba Airport Shuttle  
**Total Budget Cap:** ₹15,000  

---

## **Day 1: Coastal Forts, Dolphin Cruise & Beachside Dining**

### **Schedule & Route**
* **Morning (08:00 – 12:00): Arrival & Coastal Cruise**
  * Arrive in Goa and take the Kadamba EV AC airport shuttle to Calangute.
  * Check into **Backpacker Hostel Dorm near Calangute** and pick up your rental scooter.
  * Head down to the shore for the **Dolphin Boat Trip** (1.5-hour marine sightseeing excursion along the Sinquerim/Calangute coast).
* **Afternoon (12:30 – 16:30): Authentic Goan Lunch & Relaxation**
  * Relish an authentic **Local Fish Thali** at an acclaimed local coastal eatery.
  * Rest and unwind at the hostel or lounge along Calangute Beach.
* **Evening (16:30 – 2

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9e76ac34-3ef4-457d-847c-31fd1edc1e8d                                                                       │
│  Final Output: # 3-Day Comprehensive Goa Solo Itinerary                                                         │
│                                                                                                                 │
│  **Traveller:** 1 Solo Traveller                                                                                │
│  **Duration:** 3 Days / 3 Nights                                                                                │
│  **Base Accommodation:** Backpacker Hostel Dorm near Calangute                                                  │
│  **Primary Transit:** Self-Drive Scooter Rental + Kadamba Airport Shuttle                                       │
│  **Total Budget Cap:** ₹15,000                                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## **Day 1: Coastal Forts, Dolphin Cruise & Beachside Dining**                                                 │
│                                                                                                                 │
│  ### **Schedule & Route**                                                                                       │
│  * **Morning (08:00 – 12:00): Arrival & Coastal Cruise**                                                        │
│    * Arrive in Goa and take the Kadamba EV AC airport shuttle to Calangute.                                     │
│    * Check into **Backpacker Hostel Dorm near Calangute** and pick up your rental scooter.                      │
│    * Head down to the shore for the **Dolphin Boat Trip** (1.5-hour marine sightseeing excursion along the      │
│  Sinquerim/Calangute coast).                                                                                    │
│  * **Afternoon (12:30 – 16:30): Authentic Goan Lunch & Relaxation**                                             │
│    * Relish an authentic **Local Fish Thali** at an acclaimed local coastal eatery.                             │
│    * Rest and unwind at the hostel or lounge along Calangute Beach.                                             │
│  * **Evening (16:30 – 21:00): Heritage Sunset & Seafood Dinner**                                                │
│    * Ride to **Fort Aguada**; explore the 17th-century Portuguese coastal bastion and enjoy sunset views over   │
│  the Arabian Sea.                                                                                               │
│    * Dine on fresh catch with your feet in the sand with a **Beach Shack Seafood Dinner**.                      │
│                                                                                                                 │
│  ### **Day 1 Cost Breakdown**                                                                                   │
│  | Item / Expense | Category | Cost (INR) |                                                                     │
│  | :--- | :--- | :--- |                                                                                         │
│  | Backpacker Hostel Dorm (Night 1) | Stay | ₹700 |   

In [21]:
#Exp 6(PEFT+LoRA for text classification) Dataset-"medical_texts_hard.csv"

!rm -f medical_texts*.csv
import pandas as pd, torch
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

up = files.upload()                               # upload medical_texts.csv
df = pd.read_csv(list(up.keys())[0])
labels = sorted(df["label"].unique()); l2i = {l: i for i, l in enumerate(labels)}
df["y"] = df["label"].map(l2i)
train, test = train_test_split(df, test_size=0.25, stratify=df["y"], random_state=42)

# Pre-trained model + LoRA adapters
name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=len(labels))
cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
                 target_modules=["q_lin", "v_lin"])
model = get_peft_model(model, cfg)
model.print_trainable_parameters()                # shows how few parameters are trained

dev = "cuda" if torch.cuda.is_available() else "cpu"
model.to(dev)
enc = lambda d: {k: v.to(dev) for k, v in tok(d["text"].tolist(), truncation=True, max_length=64,
                                              padding=True, return_tensors="pt").items()}

# Training loop (only LoRA weights and the classifier head are updated)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4)
model.train()
for ep in range(8):
    tr = train.sample(frac=1, random_state=ep)
    for i in range(0, len(tr), 16):
        b = tr.iloc[i:i+16]
        loss = model(**enc(b), labels=torch.tensor(b["y"].values).to(dev)).loss
        loss.backward(); opt.step(); opt.zero_grad()
    print(f"Epoch {ep+1}  loss: {loss.item():.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    pred = model(**enc(test)).logits.argmax(1).cpu().numpy()

print("Predictions:\n")
for text, actual, p in zip(test["text"], test["y"], pred):
    print("Text     :", text)
    print("Actual   :", labels[actual])
    print("Predicted:", labels[p])
    print()

print("Accuracy:", round(accuracy_score(test["y"], pred), 3))
print(classification_report(test["y"], pred, target_names=labels))

model.save_pretrained("lora_adapter")             # saves only the small LoRA adapter   # saves only the small LoRA adapter

Saving medical_texts_hard.csv to medical_texts_hard.csv


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936
Epoch 1  loss: 1.0722
Epoch 2  loss: 1.1367
Epoch 3  loss: 0.9470
Epoch 4  loss: 0.8272
Epoch 5  loss: 0.7492
Epoch 6  loss: 0.5486
Epoch 7  loss: 0.2545
Epoch 8  loss: 0.4106
Predictions:

Text     : Complaint of pain radiating to the left arm. Findings: troponin levels are elevated. Doctor will start beta blocker therapy.
Actual   : Cardiology
Predicted: Cardiology

Text     : Patient reports dry scaly skin. Examination: dermoscopy shows irregular pigmentation. Plan: apply a topical steroid cream.
Actual   : Dermatology
Predicted: Dermatology

Text     : Complaint of recurring acne on the face. Findings: CT scan shows airway inflammation. Doctor will prescribe an inhaled bronchodilator.
Actual   : Dermatology
Predicted: Dermatology

Text     : Complaint of a changing mole on the back. Findings: silvery scales are seen on the plaques. Doctor will schedule a stress test.
Actual   : Dermatology
Predicted: Dermatol